In [ ]:
# NOTE: Run this cell after the notebook class/function definitions are loaded.
# It enforces deterministic dataset->env task binding and group-size integrity.

def _patch_notebook_for_deterministic_tasks():
    import types

    def _reset_strict(self, **kwargs):
        kwargs.pop('prompt', None)
        row_tid = kwargs.pop('episode_task_id', None)
        row_tid_s = str(row_tid).strip() if row_tid is not None else ''
        pool = [str(x) for x in (self.STAGE_TASK_IDS or [])]

        chosen_task_id = ''
        if row_tid_s and pool and row_tid_s in pool:
            chosen_task_id = row_tid_s
        elif row_tid_s and pool:
            raise ValueError(f"episode_task_id={row_tid_s!r} not in active stage pool={pool}")
        elif not row_tid_s and pool:
            raise ValueError('episode_task_id missing from dataset row; deterministic grouping requires it.')
        elif self.CURRICULUM is not None:
            chosen_task_id = self.CURRICULUM.pick_task_id(self.STAGE)
        elif self.STAGE_TASK_IDS:
            import random
            chosen_task_id = random.choice(self.STAGE_TASK_IDS)

        if self._client is not None:
            try:
                _sync(self._client.close(), timeout=10)
            except Exception:
                pass
            self._client = None
            time.sleep(0.3)
        self._client = PageZeroEnvClient(base_url=self.BASE_URL, message_timeout_s=300.0)

        try:
            result = _sync(self._client.reset(task_id=chosen_task_id)) if chosen_task_id else _sync(self._client.reset())
        except TypeError:
            result = _sync(self._client.reset())
            chosen_task_id = ''

        self.total_reward = 0.0
        self.diagnosis_reward = 0.0
        self.fix_reward = 0.0
        self.terminal_reward = 0.0
        self.is_done = bool(result.done)
        self._episode_logged = False
        self.trajectory = []
        self.task_id = chosen_task_id
        self.scenario_name = (getattr(result.observation, 'tool_output', '') or '')[:120]
        self.last_sla_status = getattr(result.observation, 'sla_status', 'OK') or 'OK'
        self.is_resolved = False
        self.stack_healthy = None
        self.start_ts = time.time()
        self.end_ts = None
        self._last_feedback = ''
        self._last_command = ''
        self._last_output_snippet = ''
        self._last_reward = 0.0

        print(f"[nb-reset] stage={self.STAGE} row_task={row_tid_s or '<missing>'} chosen={chosen_task_id or '<auto>'}")
        return self._format_observation(result.observation, reward=0.0)

    def _make_stage_dataset_strict(num_episodes: int, task_ids: list) -> Dataset:
        if int(num_episodes) < int(NUM_GENERATIONS):
            raise ValueError(
                f'NUM_EPISODES must be >= NUM_GENERATIONS; got {num_episodes} and {NUM_GENERATIONS}'
            )
        if int(num_episodes) % int(NUM_GENERATIONS) != 0:
            raise ValueError(
                f'NUM_EPISODES must be divisible by NUM_GENERATIONS; got {num_episodes} and {NUM_GENERATIONS}'
            )
        return build_grpo_dataset(int(num_episodes), list(task_ids), group_size=int(NUM_GENERATIONS))

    _orig_run_tool = PageZeroToolEnv._run_tool

    def _run_tool_with_gate_debug(self, tool, args):
        text = _orig_run_tool(self, tool, args)
        fb = str(getattr(self, '_last_feedback', '') or '')
        if 'Ignored `done`' in fb or 'documentation incomplete' in fb or 'deferred until step >=' in fb:
            print(f"  [gate-debug] {fb[:220]}")
        return text

    PageZeroToolEnv.reset = _reset_strict
    PageZeroToolEnv._run_tool = _run_tool_with_gate_debug
    globals()['make_stage_dataset'] = _make_stage_dataset_strict
    print('[patch] deterministic task binding + generation-group integrity + done/docs gate debug enabled')


_patch_notebook_for_deterministic_tasks()


# PageZero SRE Agent — GRPO Training

Trains a Qwen3-0.6B agent to diagnose and fix production incidents using GRPO.

**Stack:** TRL 1.2+ | transformers 5.x | bitsandbytes 4-bit | openenv-core

**Run all cells in order from top to bottom.**

## 1) Install Dependencies

In [4]:
import subprocess
import sys
from importlib import metadata

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'], check=True)

# Core packages — no unsloth, no vllm (they conflict with TRL 1.x / transformers 5.x)
core_packages = [
    'trl>=0.29.0',
    'transformers>=5.2.0',
    'huggingface-hub>=1.5.0,<2.0',
    'datasets',
    'accelerate',
    'bitsandbytes',
    'matplotlib',
    'pandas',
    'peft',
    'jmespath',
    'nest_asyncio',
    'liger-kernel',
    'trackio',
    'openenv-core[core]>=0.2.1',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *core_packages], check=True)

# Verify
print('-' * 40)
for pkg in ['transformers', 'trl', 'huggingface-hub', 'datasets', 'peft', 'bitsandbytes', 'liger-kernel', 'trackio']:
    try:
        print(f'{pkg}: {metadata.version(pkg)}')
    except Exception:
        print(f'{pkg}: NOT INSTALLED')

----------------------------------------
transformers: 5.6.2
trl: 1.2.0
huggingface-hub: 1.12.0
datasets: 4.8.4
peft: 0.19.1
bitsandbytes: 0.49.2
liger-kernel: 0.7.0
trackio: 0.25.0





### ⚠️ After running the cell above: Runtime → Restart runtime, then continue from Cell 2

## 2) Configuration

In [1]:
import os

IN_COLAB = 'COLAB_GPU' in os.environ

# --- HuggingFace token (optional unless pushing to hub) ---
try:
    from google.colab import userdata
    if 'HF_TOKEN' not in os.environ:
        token = userdata.get('HF_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
except Exception:
    pass

if 'HF_TOKEN' not in os.environ:
    print('WARNING: HF_TOKEN not found. Needed only for push_to_hub.')

# --- Backend mode selector ---
BACKEND_MODE = 'deployed_space'  # local_same_machine | local_via_tunnel | deployed_space | custom_url

LOCAL_SAME_MACHINE_URL = 'http://localhost:8000'
LOCAL_TUNNEL_URL = 'https://brave-pots-throw.loca.lt/'
DEPLOYED_SPACE_URL = 'https://Maruti777-pagezero-agent.hf.space'
CUSTOM_ENV_URL = os.environ.get('PAGEZERO_ENV_URL', '').strip()

if BACKEND_MODE == 'local_same_machine':
    ENV_URL = LOCAL_SAME_MACHINE_URL
elif BACKEND_MODE == 'local_via_tunnel':
    ENV_URL = LOCAL_TUNNEL_URL
elif BACKEND_MODE == 'deployed_space':
    ENV_URL = DEPLOYED_SPACE_URL
elif BACKEND_MODE == 'custom_url':
    if not CUSTOM_ENV_URL:
        raise ValueError("BACKEND_MODE='custom_url' but PAGEZERO_ENV_URL is empty.")
    ENV_URL = CUSTOM_ENV_URL
else:
    raise ValueError(f'Unknown BACKEND_MODE: {BACKEND_MODE}')

if IN_COLAB and BACKEND_MODE == 'local_same_machine':
    print('WARNING: localhost points to the Colab VM, not your laptop.')
    print('Use local_via_tunnel or deployed_space for Colab.')

# --- Repo / checkout config ---
REPO_URL = 'https://huggingface.co/spaces/pranayy/pagezero_agent'
USE_EXISTING_CHECKOUT = not IN_COLAB
EXISTING_CHECKOUT_DIR = os.getcwd() if not IN_COLAB else '/content/PageZero'
CLONE_DIR = '/content/PageZero' if IN_COLAB else EXISTING_CHECKOUT_DIR

# --- Model config ---
USE_UNSLOTH_MODEL = False  # True will try 4-bit Unsloth model id
UNSLOTH_MODEL_ID = 'unsloth/Qwen3-0.6B-bnb-4bit'
BASE_MODEL_ID = 'Qwen/Qwen3-0.6B'
MODEL_ID = UNSLOTH_MODEL_ID if USE_UNSLOTH_MODEL else BASE_MODEL_ID
FALLBACK_MODEL_ID = BASE_MODEL_ID

# --- Training hyperparameters (kube-sre-gym memory-and-speed recipe) -----
# Per-stage budget — total dataset size for the trainer.train() call. Mastery
# gates may end the stage early once the agent's recent_success_rate clears
# the tier threshold, so this is an *upper bound*, not a fixed count.
NUM_EPISODES = 30
# GRPO group g — sampled completions per prompt that compete inside a group.
# Larger g → lower advantage variance, but multiplies GPU memory and rollout
# wallclock. T4-safe = 4-6; we default to 4 to keep early iterations cheap.
NUM_GENERATIONS = 4
G_PARAM = NUM_GENERATIONS
# Tier 1 (rollout shape): allow enough tokens per *accumulated* completion so
# tool results + next <tool_call> fit; 256 caused almost all episodes to stop
# after 1 env step. Raise MAX_TOTAL_TOKENS in step with max_completion_length.
MAX_TURNS = 8
MAX_NEW_TOKENS = 1024
# Sampling — lower temperature reduces mode-collapse on diagnose_root_cause.
TEMPERATURE = 0.6
TOP_P = 0.9  # passed to GRPOConfig when the installed trl version supports top_p
# Hard cap on accumulated rollout tokens to prevent backward-pass OOM.
MAX_TOTAL_TOKENS = 12288
VLLM_MODE = 'colocate'
OUTPUT_DIR_BASE = 'outputs/pagezero-colab'

# --- KL-divergence regularization (β) ---
# Static β=0.01 (kube-sre-gym proved adaptive scheduling is unnecessary once
# the reward signal has variance). The old 0.04 over-damped exploration.
KL_BETA = 0.01

# --- Mastery-gated curriculum stages -------------------------------------
# Each stage defines a task pool. The MasteryCurriculum advances stage→stage
# only when ``recent_success_rate >= advance_rate`` over ``min_episodes``
# (kube-sre-gym pattern). ``episodes`` here is an upper-bound budget per
# stage; the gate may close the stage early on fast-track.
CURRICULUM_STAGES = [
    {
        'name': 'easy',
        'task_ids': ['task_1', 'task_2', 'task_3', 'task_4', 'task_5'],
        'episodes': NUM_EPISODES,
        'min_episodes': 5,
        'advance_rate': 0.6,
    },
    {
        'name': 'medium',
        'task_ids': ['task_6', 'task_7', 'task_8', 'task_9', 'task_10'],
        'episodes': NUM_EPISODES,
        'min_episodes': 8,
        'advance_rate': 0.65,
    },
    {
        'name': 'hard',
        'task_ids': ['task_11', 'task_12'],
        'episodes': NUM_EPISODES,
        'min_episodes': 10,
        'advance_rate': 0.7,
    },
]

# --- Trajectory + top-level reward log ----------------------------------
# Single canonical CSV at the output dir root (no per-stage subdir burying)
# plus a JSONL with the full trajectory per episode.
REWARD_LOG_NAME = 'reward_log.csv'
TRAJECTORY_LOG_NAME = 'trajectories.jsonl'

# Penalty applied when a completion parses to zero tool calls (the env was
# never stepped). Mirrors kube-sre-gym's -0.5 for invalid commands.
NO_TOOL_PENALTY = -0.5

# --- Reward audit guards ---
# Sized for kube-sre-gym-style high-variance terminals: -2.0 wipe on timeout,
# up to ~+8 on confirmed resolution. We give ~25% headroom on the upper
# bound so genuine extremes are never clipped.
REWARD_AUDIT_MIN = -3.0
REWARD_AUDIT_MAX = 10.0

# --- Hub push config ---
PUSH_TO_HUB = False
HUB_REPO = 'pranayy/pagezero_agent'

print(f'IN_COLAB        : {IN_COLAB}')
print(f'BACKEND_MODE    : {BACKEND_MODE}')
print(f'ENV_URL         : {ENV_URL}')
print(f'MODEL_ID        : {MODEL_ID}')
print(f'EPISODES/STAGE  : {NUM_EPISODES} (upper bound; mastery gate may end early)')
print(f'GENERATIONS (g) : {NUM_GENERATIONS}')
print(f'MAX_TURNS       : {MAX_TURNS}')
print(f'MAX_NEW_TOKENS  : {MAX_NEW_TOKENS}')
print(f'TEMPERATURE     : {TEMPERATURE}')
print(f'TOP_P           : {TOP_P}')
print(f'KL β            : {KL_BETA}')
print(f'CURRICULUM      : {[s["name"] for s in CURRICULUM_STAGES]}')

IN_COLAB        : True
BACKEND_MODE    : deployed_space
ENV_URL         : https://Maruti777-pagezero-agent.hf.space
MODEL_ID        : Qwen/Qwen3-0.6B
EPISODES/STAGE  : 4
GENERATIONS (g) : 8
KL β            : 0.04
CURRICULUM      : ['easy', 'medium', 'hard']


## 3) Clone Repo & Install

In [5]:
import os
import subprocess

if IN_COLAB:
    if not os.path.exists(CLONE_DIR):
        subprocess.run(['git', 'clone', REPO_URL, CLONE_DIR], check=True)
    os.chdir(CLONE_DIR)
    subprocess.run(['pip', 'install', '-q', '-e', '.[train]'], check=True)

print(f'Working directory: {os.getcwd()}')

Working directory: /content/PageZero


## 4) Test Backend Connection

In [6]:
import requests
import asyncio
import sys, os

if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

from PageZero.client import PageZeroEnvClient
from PageZero.models import PageZeroAction

print(f'Testing backend: {ENV_URL}')
url = ENV_URL.rstrip('/') + '/health'
try:
    r = requests.get(url, timeout=10)
    print(f'{url} -> {r.status_code}')
except Exception as e:
    print(f'{url} -> FAILED: {e}')

async def run_env_interaction():
    env = PageZeroEnvClient(base_url=ENV_URL, message_timeout_s=300.0)
    try:
        reset_result = await env.reset()
        obs = reset_result.observation
        print('Reset successful')
        print(f'Step {obs.step}/{obs.max_steps}')
        print(f'Active alerts: {obs.active_alerts}')

        step_result = await env.step(PageZeroAction(tool='check_alerts', args={}))
        print(f'Step reward: {step_result.reward}')
        print(f'Done: {step_result.done}')
    finally:
        await env.close()

await run_env_interaction()

Testing backend: https://Maruti777-pagezero-agent.hf.space
https://Maruti777-pagezero-agent.hf.space/health -> 200
Reset successful
Step 0/15
Active alerts: ['CRITICAL: API p99 latency > 5s, PostgreSQL CPU at 95% — investigate active queries in PostgreSQL']
Step reward: 0.115
Done: False


## 5) Import Training Utilities

In [7]:
import csv
import logging
import os
import sys
from datetime import datetime
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

from datasets import Dataset
from transformers import AutoTokenizer
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

from PageZero.train import SYSTEM_PROMPT, plot_rewards, build_grpo_dataset

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

print('Imports OK.')
print('System prompt preview:')
print(SYSTEM_PROMPT[:220] + '...')

Imports OK.
System prompt preview:
You are a Staff SRE on-call. Diagnose and fix the cascading incident across the Application, PostgreSQL, and Redis cache.

To use a tool, you MUST use this exact format:
<tool_call>
{"name": "check_alerts", "arguments": ...


## 6) Define PageZeroToolEnv (async-safe for Colab)

In [ ]:
import asyncio, threading, time, os, json
from typing import Any, Dict

from PageZero.client import PageZeroEnvClient
from PageZero.models import PageZeroAction
# Canonical wrapper + curriculum + logger live in PageZero.train so the
# notebook and CLI never drift. We subclass below only to swap the
# synchronous WebSocket client for a Colab-safe asyncio bridge.
from PageZero.train import (
    PageZeroToolEnv as _CanonicalToolEnv,
    MasteryCurriculum,
    RewardLogger,
    make_reward_total,
    reward_diagnosis_metric,
    reward_fix_metric,
)

# ── Background event loop (avoids Jupyter's "loop already running" error) ──
_bg_loop = asyncio.new_event_loop()
threading.Thread(target=_bg_loop.run_forever, daemon=True).start()


def _sync(coro, timeout=360):
    """Submit async coroutine to background loop and block until done."""
    return asyncio.run_coroutine_threadsafe(coro, _bg_loop).result(timeout=timeout)


class PageZeroToolEnv(_CanonicalToolEnv):
    """Colab-safe variant of the canonical wrapper.

    Overrides only ``__init__``, ``reset``, ``_run_tool`` and ``_close_all``
    to bridge the async openenv WebSocket client into the synchronous
    interface that TRL / GRPOTrainer expects. ALL reward/trajectory/
    feedback/curriculum logic comes from the parent class so the notebook
    and CLI behave identically.

    Curriculum hooks live on the parent as ``_set_stage`` / ``_close_all`` so TRL
    does not treat them as agent-callable tools.
    """

    BASE_URL: str = ENV_URL
    VERBOSE_REWARDS: bool = bool(int(os.environ.get('PZ_VERBOSE_REWARDS', '1')))

    def __init__(self) -> None:
        """Create a Colab-safe env wrapper without opening the sync WebSocket client."""
        # Bypass parent __init__ — it creates a sync client we don't want.
        self._client = None
        self.total_reward = 0.0
        self.diagnosis_reward = 0.0
        self.fix_reward = 0.0
        self.terminal_reward = 0.0
        self.is_done = False
        self._episode_logged = False
        self.trajectory = []
        self.task_id = ''
        self.scenario_name = ''
        self.last_sla_status = 'OK'
        self.is_resolved = False
        self.stack_healthy = None
        self.start_ts = time.time()
        self.end_ts = None
        self._last_feedback = ''
        self._last_command = ''
        self._last_output_snippet = ''
        self._last_reward = 0.0
        type(self)._instances.append(self)

    @classmethod
    def _close_all(cls) -> None:
        """Close every async WebSocket client tracked by this environment subclass.

        Returns:
            None: This is a class-level cleanup side effect only.
        """
        for env in cls._instances:
            try:
                if env._client is not None:
                    _sync(env._client.close(), timeout=10)
            except Exception:
                pass
        cls._instances.clear()

    def reset(self, **kwargs: Any) -> str | None:
        """Start a new episode and return the initial observation text for the model.

        Args:
            kwargs: Optional keyword arguments for future server reset hooks (unused here).

        Returns:
            Formatted observation string appended to the user prompt, or None if the integration passes no observation (rare).
        """
        # Tear down any prior socket and start a fresh one per episode.
        if self._client is not None:
            try:
                _sync(self._client.close(), timeout=10)
            except Exception:
                pass
            self._client = None
            time.sleep(0.3)
        self._client = PageZeroEnvClient(base_url=self.BASE_URL, message_timeout_s=300.0)

        # Pick task_id from the curriculum (preferred) or stage pool fallback.
        chosen_task_id = ''
        if self.CURRICULUM is not None:
            chosen_task_id = self.CURRICULUM.pick_task_id(self.STAGE)
        elif self.STAGE_TASK_IDS:
            import random
            chosen_task_id = random.choice(self.STAGE_TASK_IDS)

        try:
            result = _sync(self._client.reset(task_id=chosen_task_id)) if chosen_task_id \
                     else _sync(self._client.reset())
        except TypeError:
            result = _sync(self._client.reset())
            chosen_task_id = ''

        self.total_reward = 0.0
        self.diagnosis_reward = 0.0
        self.fix_reward = 0.0
        self.terminal_reward = 0.0
        self.is_done = bool(result.done)
        self._episode_logged = False
        self.trajectory = []
        self.task_id = chosen_task_id
        self.scenario_name = (getattr(result.observation, 'tool_output', '') or '')[:120]
        self.last_sla_status = getattr(result.observation, 'sla_status', 'OK') or 'OK'
        self.is_resolved = False
        self.stack_healthy = None
        self.start_ts = time.time()
        self.end_ts = None
        self._last_feedback = ''
        self._last_command = ''
        self._last_output_snippet = ''
        self._last_reward = 0.0

        if self.VERBOSE_REWARDS:
            print(f'[env-reset] stage={self.STAGE} task={chosen_task_id or "auto"} '
                  f'tier={self.CURRICULUM.get_tier_name() if self.CURRICULUM else "n/a"}')
        return self._format_observation(result.observation, reward=0.0)

    def _run_tool(self, tool: str, args: Dict[str, Any]) -> str:
        """Execute one tool call via the async WebSocket client (internal bridge)."""
        if self.is_done and tool != 'done':
            raise ValueError('Episode already done.')

        # One reconnect retry on transient WebSocket drops.
        for attempt in range(2):
            try:
                result = _sync(self._client.step(PageZeroAction(tool=tool, args=args)))
                break
            except Exception as e:
                if attempt == 0 and 'Closed' in type(e).__name__:
                    print('  [reconnect] WebSocket dropped, reconnecting...')
                    self._client = PageZeroEnvClient(base_url=self.BASE_URL,
                                                    message_timeout_s=300.0)
                    _sync(self._client.reset())
                    continue
                raise

        reward = float(result.reward or 0.0)
        was_done = bool(result.done)
        obs = result.observation

        canonical_final = getattr(obs, 'final_score', None)
        try:
            canonical_final = float(canonical_final) if canonical_final is not None else None
        except Exception:
            canonical_final = None

        per_step, terminal = self._split_terminal(reward, was_done, canonical_final)
        if was_done:
            self.terminal_reward = terminal

        self.total_reward += reward
        self.is_done = was_done

        # Trust the env's own is_fix_step flag (set by the auto-detect-fix path).
        is_fix = bool(getattr(obs, 'is_fix_step', False)) or tool in (
            'pg_cancel_query', 'pg_create_index', 'pg_vacuum',
            'redis_flush_db', 'docker_restart', 'rollback_deploy',
        )
        if is_fix:
            self.fix_reward += per_step
        else:
            self.diagnosis_reward += per_step

        sla_status = getattr(obs, 'sla_status', 'OK') or 'OK'
        self.last_sla_status = sla_status
        snippet = (getattr(obs, 'tool_output', '') or '')[:240]

        feedback = getattr(obs, 'judge_feedback', None) or getattr(obs, 'hint', '') or ''
        self._last_command = json.dumps({'name': tool, 'arguments': args}, default=str)[:200]
        self._last_feedback = str(feedback or '')
        self._last_output_snippet = snippet
        self._last_reward = reward

        self.trajectory.append({
            'step': len(self.trajectory) + 1,
            'tool': tool,
            'args': args,
            'reward': reward,
            'per_step_reward': per_step,
            'terminal_reward_component': terminal if was_done else 0.0,
            'is_fix': is_fix,
            'phase': getattr(obs, 'phase', None),
            'repeat_count': getattr(obs, 'repeat_count', 1),
            'sla_status': sla_status,
            'is_done': was_done,
            'stack_healthy': getattr(obs, 'stack_healthy', None),
            'judge_feedback': str(feedback or '')[:300],
            'output_snippet': snippet,
        })

        if was_done:
            self.end_ts = time.time()
            self.stack_healthy = getattr(obs, 'stack_healthy', None)
            self.is_resolved = bool(self.stack_healthy)

        if self.VERBOSE_REWARDS:
            tag = 'FIX' if is_fix else 'DIAG'
            term_note = f' term={terminal:+.3f}' if was_done else ''
            print(
                f'  [step {len(self.trajectory):>2}] {tag:<4s} {tool:<22s} '
                f'r={reward:+.3f} cum={self.total_reward:+.3f}{term_note}'
            )

        return self._format_observation(obs, reward=reward)


# Wire ENV_URL into the canonical wrapper class attribute.
PageZeroToolEnv.BASE_URL = ENV_URL
PageZeroToolEnv.NO_TOOL_PENALTY = NO_TOOL_PENALTY


# ── Quick smoke test ──────────────────────────────────────────────────────
print('Smoke-testing PageZeroToolEnv...')
_test_env = PageZeroToolEnv()
_test_obs = _test_env.reset()
print(f'Reset OK. Observation preview:\n{_test_obs[:300]}...')
_sync(_test_env._client.close(), timeout=10)
PageZeroToolEnv._instances.clear()
print('Smoke test PASSED ✓')


## 7) Build GRPO Trainer

In [ ]:
import torch, gc, csv, json, math
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig
from datasets import Dataset
from transformers import AutoTokenizer
from pathlib import Path
from datetime import datetime

# --- 1. PREVENT OOM on re-runs ---
try:
    if 'trainer' in locals():
        del trainer
    if 'model' in locals():
        del model
    gc.collect()
    torch.cuda.empty_cache()
except Exception:
    pass

# Help PyTorch reuse fragmented GPU memory (kube-sre-gym recipe).
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# --- 2. Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 3. Output dir + canonical top-level reward logger -------------------
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
output_dir = Path(f'{OUTPUT_DIR_BASE}-{timestamp}')
output_dir.mkdir(parents=True, exist_ok=True)

# Single top-level reward_log.csv + trajectories.jsonl (no per-stage burying).
reward_logger = RewardLogger(output_dir, csv_name=REWARD_LOG_NAME,
                             jsonl_name=TRAJECTORY_LOG_NAME)

# Mastery-gated curriculum shared across stages.
stage_pools = {s['name']: list(s['task_ids']) for s in CURRICULUM_STAGES}
mastery_curriculum = MasteryCurriculum(stage_task_pools=stage_pools)

# Wire the wrapper to the logger + curriculum (class-level so every new
# environment_factory() call picks them up).
PageZeroToolEnv.REWARD_LOGGER = reward_logger
PageZeroToolEnv.CURRICULUM = mastery_curriculum

print(f'Output dir       : {output_dir}')
print(f'Reward log       : {reward_logger.csv_path}')
print(f'Trajectory log   : {reward_logger.jsonl_path}')
print(f'Curriculum tier  : {mastery_curriculum.get_tier_name()}')

# --- 4. Reward funcs (single primary; diag/fix kept as logging metrics) --
# TRL sums all reward_funcs by default — passing diagnosis/fix/total caused
# double-counting in earlier runs. We pass *only* reward_total. The other
# two are still computed by the wrapper and surfaced in the CSV.
reward_total = make_reward_total(no_tool_penalty=NO_TOOL_PENALTY)

def _audit(values, label):
    """Clip / sanitize a reward batch and emit per-call stats."""
    cleaned = []
    for v in values:
        try:
            v = float(v)
        except Exception:
            v = 0.0
        if math.isnan(v) or math.isinf(v):
            print(f'  [reward-audit:{label}] WARN non-finite reward → 0.0')
            v = 0.0
        if v < REWARD_AUDIT_MIN or v > REWARD_AUDIT_MAX:
            print(f'  [reward-audit:{label}] WARN clip {v:+.3f} → '
                  f'[{REWARD_AUDIT_MIN}, {REWARD_AUDIT_MAX}]')
            v = max(REWARD_AUDIT_MIN, min(REWARD_AUDIT_MAX, v))
        cleaned.append(v)
    if cleaned:
        mu = sum(cleaned) / len(cleaned)
        var = sum((x - mu) ** 2 for x in cleaned) / max(1, len(cleaned) - 1)
        sd = var ** 0.5
        print(f'  [reward-audit:{label}] n={len(cleaned)} mean={mu:+.3f} '
              f'std={sd:.3f} min={min(cleaned):+.3f} max={max(cleaned):+.3f}')
    return cleaned

def reward_total_audited(completions=None, environments=None, **kwargs):
    return _audit(reward_total(completions=completions,
                               environments=environments, **kwargs),
                  'total')

# --- 5. Dataset (per-row EPISODE_BRIEF + episode_task_id for TRL→reset) ---
def make_stage_dataset(num_episodes: int, task_ids: list) -> Dataset:
    return build_grpo_dataset(int(num_episodes), list(task_ids))

dataset = make_stage_dataset(NUM_EPISODES, CURRICULUM_STAGES[0]['task_ids'])

# --- 6. Debug print: confirm chat template + tool list reach the model ---
# Many "0 tool calls" episodes were caused by the chat template silently
# wrapping completions in <think>...</think> blocks. Lock enable_thinking
# off and print one full prompt so we can eyeball the formatting.
_dbg_prompt = dataset[0]['prompt']
try:
    debug_prompt = tokenizer.apply_chat_template(
        _dbg_prompt,
        add_generation_prompt=True,
        tokenize=False,
        enable_thinking=False,
    )
except TypeError:
    debug_prompt = tokenizer.apply_chat_template(
        _dbg_prompt,
        add_generation_prompt=True,
        tokenize=False,
    )
print('=== DEBUG: rendered chat-template prompt (first 1200 chars) ===')
print(debug_prompt[:1200])
print('=== END DEBUG ===')

# --- 7. Load model with 4-bit quantization (T4-safe) --------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

# --- 8. GRPO Config ------------------------------------------------------
grpo_kwargs = dict(
    use_vllm=False,
    output_dir=str(output_dir),
    num_train_epochs=1,
    learning_rate=2e-6,
    lr_scheduler_type='cosine',
    max_tool_calling_iterations=MAX_TURNS,
    warmup_steps=2,
    max_grad_norm=1.0,
    gradient_accumulation_steps=1,
    per_device_train_batch_size=1,
    generation_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_NEW_TOKENS,
    logging_steps=1,
    save_strategy='steps',
    save_steps=10,
    temperature=TEMPERATURE,
    report_to='trackio',
    run_name=f'pagezero-grpo-{timestamp}',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    save_total_limit=3,
    loss_type='dapo',
    mask_truncated_completions=False,
    beta=KL_BETA,
    # Critical: disable Qwen3 chain-of-thought wrapper so completions parse
    # to <tool_call> blocks instead of <think>...</think> blobs.
    chat_template_kwargs={'enable_thinking': False},
)

grpo_fields = set(getattr(GRPOConfig, '__dataclass_fields__', {}).keys())
if 'use_liger_kernel' in grpo_fields:
    grpo_kwargs['use_liger_kernel'] = True
if 'project' in grpo_fields:
    grpo_kwargs['project'] = 'pagezero-rl'
if 'top_p' in grpo_fields:
    grpo_kwargs['top_p'] = TOP_P

grpo_config = GRPOConfig(**grpo_kwargs)

# --- 9. LoRA Config ------------------------------------------------------
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

# --- 10. Initialize Trainer (single primary reward) ---------------------
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_total_audited],
    train_dataset=dataset,
    args=grpo_config,
    peft_config=peft_config,
    environment_factory=PageZeroToolEnv,
)

print(f'Trainer initialized. Output path: {output_dir}')
print(f'  KL β             : {KL_BETA}')
print(f'  num_generations  : {NUM_GENERATIONS}  (g)')
print(f'  temperature      : {TEMPERATURE}')
print(f'  top_p            : {TOP_P} (if supported by GRPOConfig)')
print(f'  max_completion_length: {MAX_NEW_TOKENS}')
print(f'  mask_truncated   : False (Tier 1 — keep truncated rollouts in the loss)')
print(f'  enable_thinking  : False (forced by chat_template_kwargs)')
print(f'  reward_funcs     : [reward_total_audited] (single primary)')
print(f'  curriculum stages: {[s["name"] for s in CURRICULUM_STAGES]}')
print(f'  reward log       : {reward_logger.csv_path}')
print(f'  trajectory log   : {reward_logger.jsonl_path}')


## 8) Train, Plot, and Save

In [ ]:
PUSH_TO_HUB = True

## 8.0) Baseline Evaluation (pre-training)

Captures task-by-task performance of the **base** model so the final eval cell below can show a clean baseline-vs-trained delta.

Outputs `<output_dir>/baseline_eval.json` (consumed by the submission plot generator).

In [ ]:
import os, sys, subprocess
from pathlib import Path

EVAL_EPISODES_PER_TASK = int(os.environ.get('EVAL_EPISODES_PER_TASK', '3'))
EVAL_TASKS = ['task_1', 'task_2', 'task_3', 'task_4', 'task_5']

baseline_path = output_dir / 'baseline_eval.json'
script_path = Path(CLONE_DIR) / 'scripts' / 'eval_checkpoint.py'

cmd = [
    sys.executable, str(script_path),
    '--base-model', MODEL_ID,
    '--env-url', ENV_URL,
    '--output', str(baseline_path),
    '--label', 'baseline',
    '--tasks', *EVAL_TASKS,
    '--episodes-per-task', str(EVAL_EPISODES_PER_TASK),
    '--max-turns', str(MAX_TURNS),
    '--max-new-tokens', str(min(MAX_NEW_TOKENS, 512)),
]
print('[baseline-eval] running:')
print('  ' + ' '.join(cmd))
proc = subprocess.run(cmd)
if proc.returncode != 0:
    print(f'WARNING: baseline eval exit={proc.returncode}; submission plots will fall back to empty baseline.')
else:
    print(f'[baseline-eval] wrote {baseline_path}')

In [ ]:
import json, gc
import pandas as pd
from IPython.display import display, Image
from PageZero.train import plot_rewards

print('Starting GRPO curriculum training (mastery-gated)...')
print(f'  Model        : {MODEL_ID}')
print(f'  KL β         : {KL_BETA}')
print(f'  g (num_gen)  : {NUM_GENERATIONS}')
print(f'  Episodes/stg : {NUM_EPISODES} (upper bound; gate may end early)')
print(f'  Environment  : {ENV_URL}')
print(f'  Stages       : {[s["name"] for s in CURRICULUM_STAGES]}')
print()


# --- Curriculum loop -----------------------------------------------------
# The MasteryCurriculum tracks recent_success_rate over a sliding window;
# advancement happens *inside* the wrapper as soon as recent_success_rate
# clears the tier's advance_rate. Here we simply iterate over the stages
# and let the trainer run up to NUM_EPISODES rollouts per stage.
STAGE_CHECKPOINTS: dict[str, "Path"] = {}
STAGE_SUMMARIES: dict[str, dict] = {}


def _summarize(jsonl_path, stage):
    """Aggregate trajectory-level stats for one stage from the global JSONL."""
    if not jsonl_path.exists():
        return {}
    rows = []
    with open(jsonl_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except Exception:
                continue
            if row.get('stage') == stage:
                rows.append(row)
    if not rows:
        return {}
    totals = [float(r.get('total_reward', 0.0)) for r in rows]
    norms = [float(r.get('normalized_reward', 0.0)) for r in rows]
    resolved = [bool(r.get('is_resolved')) for r in rows]
    n_steps = [int(r.get('num_steps', 0)) for r in rows]
    tool_freq = {}
    for r in rows:
        for t in r.get('tool_sequence', []):
            tool_freq[t] = tool_freq.get(t, 0) + 1
    return {
        'episodes': len(rows),
        'mean_reward': sum(totals) / len(totals),
        'mean_normalized': sum(norms) / len(norms),
        'best_reward': max(totals),
        'worst_reward': min(totals),
        'resolution_rate': sum(resolved) / len(resolved),
        'mean_steps': sum(n_steps) / len(n_steps),
        'top_tools': sorted(tool_freq.items(), key=lambda kv: -kv[1])[:5],
    }


try:
    for stage in CURRICULUM_STAGES:
        stage_name = stage['name']
        # Bind the stage so the wrapper samples task ids from the right pool
        # (the curriculum may *also* bias toward weak spots inside that pool).
        PageZeroToolEnv._set_stage(stage_name, stage['task_ids'])

        # Refresh dataset for this stage; reset trainer epoch state so
        # repeated trainer.train() calls behave like clean stages.
        trainer.train_dataset = make_stage_dataset(stage['episodes'], stage['task_ids'])
        stage_dir = output_dir / f'stage_{stage_name}'
        stage_dir.mkdir(parents=True, exist_ok=True)
        trainer.args.output_dir = str(stage_dir)
        trainer.args.run_name = f'pagezero-grpo-{timestamp}-{stage_name}'
        trainer.state.epoch = 0
        if hasattr(trainer.state, 'global_step'):
            trainer.state.global_step = 0
        if hasattr(trainer, '_episodes_completed'):
            trainer._episodes_completed = 0

        print('=' * 70)
        print(f'[curriculum] STAGE = {stage_name.upper()}  '
              f'(episodes_budget={stage["episodes"]}, β={KL_BETA}, '
              f'g={NUM_GENERATIONS}, tasks={stage["task_ids"]})')
        print(f'[curriculum] tier={mastery_curriculum.get_tier_name()} '
              f'recent_success_rate={mastery_curriculum._recent_success_rate():.0%}')
        print('=' * 70)

        try:
            trainer.train()
        finally:
            PageZeroToolEnv._close_all()

        # Save a per-stage transition checkpoint for rollback / comparison.
        ckpt_dir = stage_dir / 'checkpoint'
        ckpt_dir.mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(ckpt_dir))
        STAGE_CHECKPOINTS[stage_name] = ckpt_dir
        print(f'[curriculum] {stage_name} → checkpoint saved at {ckpt_dir}')

        summary = _summarize(reward_logger.jsonl_path, stage_name)
        STAGE_SUMMARIES[stage_name] = summary
        if summary:
            print(
                f'[curriculum] {stage_name} summary: '
                f'episodes={summary["episodes"]} '
                f'mean={summary["mean_reward"]:+.3f} '
                f'normalized={summary["mean_normalized"]:+.3f} '
                f'best={summary["best_reward"]:+.3f} '
                f'resolved={summary["resolution_rate"]:.0%} '
                f'avg_steps={summary["mean_steps"]:.1f} '
                f'tier_after={mastery_curriculum.get_tier_name()}'
            )

        gc.collect()
        torch.cuda.empty_cache()

finally:
    PageZeroToolEnv._close_all()


# --- Final post-curriculum save + reporting -----------------------------
final_dir = output_dir / 'final'
final_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(final_dir))
print(f'\nFinal model saved to {final_dir}')

# Single canonical reward plot from the top-level CSV.
try:
    plot_rewards(reward_logger.csv_path, output_dir / 'reward_plot.png')
    if (output_dir / 'reward_plot.png').exists():
        display(Image(filename=str(output_dir / 'reward_plot.png')))
except Exception as e:
    print(f'WARNING: could not plot rewards: {e}')

print('\n=== Curriculum stage checkpoints ===')
for name, path in STAGE_CHECKPOINTS.items():
    print(f'  {name:6s} → {path}')

print('\n=== Per-stage trajectory summary ===')
summary_rows = []
for name, summary in STAGE_SUMMARIES.items():
    if not summary:
        continue
    summary_rows.append({
        'stage': name,
        'episodes': summary['episodes'],
        'mean_reward': round(summary['mean_reward'], 3),
        'normalized_mean': round(summary['mean_normalized'], 3),
        'best_reward': round(summary['best_reward'], 3),
        'worst_reward': round(summary['worst_reward'], 3),
        'resolution_rate': round(summary['resolution_rate'], 3),
        'mean_steps': round(summary['mean_steps'], 2),
    })
if summary_rows:
    display(pd.DataFrame(summary_rows))

# Show last N rows of the canonical reward log for a quick sanity check.
if reward_logger.csv_path.exists():
    df = pd.read_csv(reward_logger.csv_path)
    display(df.tail(15))

if PUSH_TO_HUB:
    if 'HF_TOKEN' not in os.environ:
        raise RuntimeError('HF_TOKEN not set. Add it in Colab secrets or env vars.')
    trainer.push_to_hub(repo_id=HUB_REPO)
    print(f'Model pushed to https://huggingface.co/{HUB_REPO}')
else:
    print('Skipping push. Set PUSH_TO_HUB=True to enable.')


## 9) Final Evaluation + Submission Plots

Re-evaluates the trained adapter on the same task pool / seed offset that the baseline used, then generates the 5 submission plots from `reward_log.csv` + the two eval JSONs.

Outputs (under `<output_dir>/`):
- `final_eval.json`
- `plots/plot_01_overall_reward_curve.png`
- `plots/plot_02_resolved_rate_curve.png`
- `plots/plot_03_taskwise_baseline_vs_trained_reward.png`
- `plots/plot_04_taskwise_baseline_vs_trained_success.png`
- `plots/plot_05_termination_reason_distribution.png`

In [ ]:
import os, sys, subprocess
from pathlib import Path
from IPython.display import display, Image, Markdown

EVAL_EPISODES_PER_TASK = int(os.environ.get('EVAL_EPISODES_PER_TASK', '3'))
EVAL_TASKS = ['task_1', 'task_2', 'task_3', 'task_4', 'task_5']

final_eval_path = output_dir / 'final_eval.json'
trained_adapter = str(final_dir)  # set in the training cell above
script_path = Path(CLONE_DIR) / 'scripts' / 'eval_checkpoint.py'
plot_script = Path(CLONE_DIR) / 'scripts' / 'generate_submission_plots.py'

eval_cmd = [
    sys.executable, str(script_path),
    '--base-model', MODEL_ID,
    '--adapter', trained_adapter,
    '--env-url', ENV_URL,
    '--output', str(final_eval_path),
    '--label', 'trained',
    '--tasks', *EVAL_TASKS,
    '--episodes-per-task', str(EVAL_EPISODES_PER_TASK),
    '--max-turns', str(MAX_TURNS),
    '--max-new-tokens', str(min(MAX_NEW_TOKENS, 512)),
]
print('[final-eval] running:')
print('  ' + ' '.join(eval_cmd))
proc = subprocess.run(eval_cmd)
if proc.returncode != 0:
    print(f'WARNING: final eval exit={proc.returncode}')

plot_cmd = [
    sys.executable, str(plot_script),
    '--run-dir', str(output_dir),
]
print('[submission-plots] running:')
print('  ' + ' '.join(plot_cmd))
subprocess.run(plot_cmd)

display(Markdown('### Submission plots'))
plot_dir = output_dir / 'plots'
plot_files = [
    'plot_01_overall_reward_curve.png',
    'plot_02_resolved_rate_curve.png',
    'plot_03_taskwise_baseline_vs_trained_reward.png',
    'plot_04_taskwise_baseline_vs_trained_success.png',
    'plot_05_termination_reason_distribution.png',
]
for name in plot_files:
    p = plot_dir / name
    if p.exists():
        display(Markdown(f'**{name}**'))
        display(Image(filename=str(p)))
    else:
        print(f'(missing) {p}')

print('\nArtifacts ready at:', output_dir)
print('  reward_log.csv          :', (output_dir / 'reward_log.csv').exists())
print('  trajectories.jsonl      :', (output_dir / 'trajectories.jsonl').exists())
print('  baseline_eval.json      :', (output_dir / 'baseline_eval.json').exists())
print('  final_eval.json         :', final_eval_path.exists())
print('  plots/                  :', plot_dir.exists() and any(plot_dir.iterdir()))

In [11]:
import shutil

archive_name = output_dir.parent / f'{output_dir.name}.zip'
shutil.make_archive(str(archive_name.with_suffix('')), 'zip', output_dir)
print(f'Archived output folder to: {archive_name}')

Archived output folder to: outputs/pagezero-colab-2026-04-25_19-41-06.zip


In [ ]:
import subprocess
import os

log_file_path = '/content/PageZero/server.log'
if os.path.exists(log_file_path):
    try:
        result = subprocess.run(['cat', log_file_path], capture_output=True, text=True, check=True)
        print(result.stdout)
    except subprocess.CalledProcessError as e:
        print(f"Error reading log file: {e}")
        print(f"Stderr: {e.stderr}")
    except FileNotFoundError:
        print(f"Log file not found at {log_file_path}")
else:
    print(f"Log file not found at {log_file_path}")

## 8) Latest Run Snapshot (2026-04-25)

**Run config**
- Model: `Qwen/Qwen3-1.7B`
- Episodes: `6`
- Generations: `6`
- Environment: `https://pranayy-pagezero-agent.hf.space/`
- Runtime: `36/36` steps in `46:43` (Epoch `1/1`)

**Training loss trace (steps 1-36)**

```text
1: 0.000000, 2: 0.228079, 3: -0.283213, 4: 0.005372, 5: 0.000000, 6: 0.000000,
7: 0.000000, 8: 0.000000, 9: 0.000000, 10: 0.062890, 11: 0.000000, 12: -0.297732,
13: 0.000000, 14: 0.000002, 15: 0.000001, 16: 0.000000, 17: 0.000000, 18: 0.000003,
19: 0.000000, 20: 0.000000, 21: 0.000004, 22: 0.000000, 23: 0.000000, 24: 0.000000,
25: 0.000000, 26: 0.000000, 27: 0.000000, 28: 0.000000, 29: 0.000000, 30: 0.000000,
31: 0.020289, 32: 0.281001, 33: -0.190548, 34: 0.000000, 35: -0.202171, 36: 0.000000
```

**Saved model**
- `outputs/pagezero-colab-2026-04-25_14-46-12`

**Reward curve summary (from plot)**
- Episodes: `36`
- Final average reward: `0.02`
- Best episode reward: `0.40`
- Trend: slightly positive, but near-flat and high-variance.

**Reward log tail**

| episode | total_reward | diagnosis_reward | fix_reward | timestamp |
|---:|---:|---:|---:|---|
| 27 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537719 |
| 28 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537743 |
| 29 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537767 |
| 30 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537790 |
| 31 | 0.2 | 0.0 | 0.2 | 2026-04-25T15:53:59.121829 |
| 32 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.121913 |
| 33 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.121943 |
| 34 | -0.2 | 0.0 | -0.2 | 2026-04-25T15:53:59.121967 |
| 35 | 0.2 | 0.2 | 0.0 | 2026-04-25T15:53:59.121991 |
| 36 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.122014 |


Latest run log placeholder

## Latest Run Snapshot (2026-04-25)

- Model: `Qwen/Qwen3-1.7B`
- Episodes: `6`
- Generations: `6`
- Environment: `https://pranayy-pagezero-agent.hf.space/`
- Runtime: `36/36` steps in `46:43`
- Saved model: `outputs/pagezero-colab-2026-04-25_14-46-12`
- Reward summary: episodes `36`, final avg `0.02`, best `0.40`.

## 8) Latest Run Snapshot (2026-04-25)

**Run config**
- Model: `Qwen/Qwen3-1.7B`
- Episodes: `6`
- Generations: `6`
- Environment: `https://pranayy-pagezero-agent.hf.space/`
- Runtime: `36/36` steps in `46:43` (Epoch `1/1`)

**Training loss trace (steps 1-36)**

```text
1: 0.000000, 2: 0.228079, 3: -0.283213, 4: 0.005372, 5: 0.000000, 6: 0.000000,
7: 0.000000, 8: 0.000000, 9: 0.000000, 10: 0.062890, 11: 0.000000, 12: -0.297732,
13: 0.000000, 14: 0.000002, 15: 0.000001, 16: 0.000000, 17: 0.000000, 18: 0.000003,
19: 0.000000, 20: 0.000000, 21: 0.000004, 22: 0.000000, 23: 0.000000, 24: 0.000000,
25: 0.000000, 26: 0.000000, 27: 0.000000, 28: 0.000000, 29: 0.000000, 30: 0.000000,
31: 0.020289, 32: 0.281001, 33: -0.190548, 34: 0.000000, 35: -0.202171, 36: 0.000000
```

**Saved model**
- `outputs/pagezero-colab-2026-04-25_14-46-12`

**Reward curve summary (from plot)**
- Episodes: `36`
- Final average reward: `0.02`
- Best episode reward: `0.40`
- Trend: slightly positive, but near-flat and high-variance.

**Reward log tail**

| episode | total_reward | diagnosis_reward | fix_reward | timestamp |
|---:|---:|---:|---:|---|
| 27 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537719 |
| 28 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537743 |
| 29 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537767 |
| 30 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:45:12.537790 |
| 31 | 0.2 | 0.0 | 0.2 | 2026-04-25T15:53:59.121829 |
| 32 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.121913 |
| 33 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.121943 |
| 34 | -0.2 | 0.0 | -0.2 | 2026-04-25T15:53:59.121967 |
| 35 | 0.2 | 0.2 | 0.0 | 2026-04-25T15:53:59.121991 |
| 36 | 0.0 | 0.0 | 0.0 | 2026-04-25T15:53:59.122014 |

